In [1]:
using Pkg
Pkg.activate(".")

  Activating project at `~/Projects/2025-Tablacares_popgen/FST`


In [4]:
using PopGen
using CSV
using DataFrames
using TidierData

Read in the all the loci

In [13]:
yft = PopGen.read("../data/YFT.snp.kinrm.pcrelate.gen")
newpops = ["ATL", "GOA", "IVC", "SEN", "VZ"]
populations!(yft, newpops)
populations(yft, counts=true)

┌ Info: 
│  /home/pdimens/Projects/2025-Tablacares_popgen/data/YFT.snp.kinrm.pcrelate.gen
│  formatting: delimiter = tab, loci = vertical
└  data: loci = 7910, samples = 417, populations = 5



 Renaming unique populations



Row,population,count
,String,Int64
1,ATL,77
2,GOA,118
3,IVC,72
4,SEN,68
5,VZ,82


Global summary stats

In [14]:
lxl_global = summary(yft, by = "locus")
CSV.write("global.nei.fst", lxl_global)

"global.nei.fst"

Perform a locus-by-locus FST (Nei) against populations

In [16]:
fst_lxl = pairwisefst(yft, by="locus", method = Nei) ;
dropmissing!(fst_lxl.results, disallowmissing=true) ;
delete!(fst_lxl.results, isnan.(fst_lxl.results.fst)) ;
CSV.write("locbyloc.nei.fst", fst_lxl.results)

"locbyloc.nei.fst"

Split the outlier and neutral data

In [17]:
f = open("../outliers/bscan_outflank.outliers")    
outlier_loc = [line for line in readlines(f)]
close(f)
outlier_loc

11-element Vector{String}:
 "Talbacares_contig_3974_68999"
 "Talbacares_contig_4875_8668"
 "Talbacares_contig_5081_16094"
 "Talbacares_contig_5081_299323"
 "Talbacares_contig_5081_299511"
 "Talbacares_contig_5081_299524"
 "Talbacares_lg_4_3319997"
 "Talbacares_lg_4_3951863"
 "Talbacares_lg_11_7575958"
 "Talbacares_lg_17_3436257"
 "Talbacares_lg_22_772470"

Split outliers into their own Popdata object

In [18]:
yft_outliers = keep(yft, locus = outlier_loc)
yft_outliers

PopData{Diploid, 11 SNP loci}
  Samples: 417
  Populations: 5

Remove outliers (in-place) from original PopData

In [19]:
omit!(yft, locus = outlier_loc)
yft

PopData{Diploid, 7899 SNP loci}
  Samples: 417
  Populations: 5

FST on outlier data. Doing this first because it's faster and will precompile for the neutral data after.

In [20]:
fst_outlier = pairwisefst(yft_outliers, iterations=75000) ;
CSV.write("amovafst.outlier.csv", fst_outlier.results)

In [21]:
fst_outlier.results

Row,ATL,GOA,IVC,SEN,VZ
,Float64,Float64,Float64,Float64,Float64
1,0.0,1.33333e-5,0.0008,5.33333e-5,0.01352
2,0.0907796,0.0,1.33333e-5,1.33333e-5,0.000106667
3,0.0655644,0.224413,0.0,0.0875467,1.33333e-5
4,0.114338,0.309495,0.0148851,0.0,1.33333e-5
5,0.0302529,0.0531962,0.164721,0.227257,0.0


FST on the neutral data

In [22]:
fst_neutral = pairwisefst(yft, iterations = 75000) ;
CSV.write("amovafst.neutral.csv", fst_neutral.results)

In [23]:
fst_neutral.results

Row,ATL,GOA,IVC,SEN,VZ
,Float64,Float64,Float64,Float64,Float64
1,0.0,0.0621333,1.33333e-5,0.124413,0.134053
2,0.00173478,0.0,2.66667e-5,0.0420533,0.520747
3,0.0113864,0.00550695,0.0,1.33333e-5,1.33333e-5
4,0.00138525,0.00151879,0.00557741,0.0,0.0747733
5,0.00119452,-0.000186736,0.00646351,0.00138338,0.0
